# Parameter Shift: Efficient Contraction vs Right Suffix Sampling

Compare accuracy and wall-clock time of the two batched parameter-shift backends:
- **EFFICIENT_CONTRACTION** — exact, O(chi^5) Kronecker contraction
- **RIGHT_SUFFIX_SAMPLING** — Monte Carlo via right-suffix sampling

In [ ]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt
from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.optimization.gradients.batch_parameter_shift import (
    expectation_value_batch_efficient_contraction,
    expectation_value_batch_right_suffix,
    BatchParameterShiftGradient,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Helpers

In [ ]:
def build_hea_circuit(N, rank, depth, device=DEVICE):
    """Hardware-efficient ansatz: RY-RZ per qubit + alternating CX layers."""
    c = Circuit(num_qubit=N, rank=rank, device=device)
    for d in range(depth):
        for q in range(N):
            c.ry(q)
            c.rz(q)
        for q in range(0, N - 1, 2):
            c.cx(q, q + 1)
        for q in range(1, N - 1, 2):
            c.cx(q, q + 1)
    return c


def build_tfim_hamiltonian(N, J=1.0, h=0.5):
    """Transverse-field Ising model: H = -J sum ZZ - h sum X."""
    ham = Hamiltonian(num_qubits=N)
    for i in range(N - 1):
        pauli = ['I'] * N
        pauli[i] = 'Z'
        pauli[i + 1] = 'Z'
        ham.add_pauli(''.join(pauli), -J)
    for i in range(N):
        pauli = ['I'] * N
        pauli[i] = 'X'
        ham.add_pauli(''.join(pauli), -h)
    return ham


def time_expectation(fn, param_batch, circuit, hamiltonian, shots, warmup=2, repeats=5):
    """Time a batched expectation function. Returns (result, median_time_ms)."""
    for _ in range(warmup):
        fn(param_batch, circuit, hamiltonian, shots)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

    times = []
    for _ in range(repeats):
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = fn(param_batch, circuit, hamiltonian, shots)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return result, np.median(times)


def time_gradient(grad_obj, theta, circuit, hamiltonian, warmup=1, repeats=3):
    """Time a full gradient computation. Returns (grad, median_time_ms)."""
    for _ in range(warmup):
        grad_obj.run(theta, circuit, hamiltonian)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

    times = []
    for _ in range(repeats):
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        result = grad_obj.run(theta, circuit, hamiltonian)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return result, np.median(times)

## 1. Expectation value accuracy vs depth

Compare both methods against each other as circuit depth increases (where SVD truncation matters).

In [ ]:
N = 8
rank = 4
depths = [1, 2, 5, 10, 20, 30]
shots = 100_000

results = []
for depth in depths:
    circuit = build_hea_circuit(N, rank, depth, device=DEVICE)
    hamiltonian = build_tfim_hamiltonian(N)
    theta = torch.randn(circuit.params_size, device=DEVICE)
    param_batch = theta.unsqueeze(0)  # (1, P)

    val_ec = expectation_value_batch_efficient_contraction(param_batch, circuit, hamiltonian, shots).item()

    # Average multiple RSS runs for stability
    rss_vals = []
    for _ in range(5):
        v = expectation_value_batch_right_suffix(param_batch, circuit, hamiltonian, shots).item()
        rss_vals.append(v)
    val_rss = np.mean(rss_vals)
    val_rss_std = np.std(rss_vals)

    results.append({
        'depth': depth,
        'ec': val_ec,
        'rss_mean': val_rss,
        'rss_std': val_rss_std,
        'diff': abs(val_ec - val_rss),
    })
    print(f"depth={depth:3d}  EC={val_ec:+.6f}  RSS={val_rss:+.6f} +/- {val_rss_std:.6f}  |diff|={abs(val_ec - val_rss):.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ds = [r['depth'] for r in results]
ec_vals = [r['ec'] for r in results]
rss_vals = [r['rss_mean'] for r in results]
rss_stds = [r['rss_std'] for r in results]
diffs = [r['diff'] for r in results]

ax = axes[0]
ax.plot(ds, ec_vals, 'o-', label='Efficient Contraction (exact)')
ax.errorbar(ds, rss_vals, yerr=rss_stds, fmt='s--', label=f'RSS ({shots//1000}k shots)', capsize=3)
ax.set_xlabel('HEA Depth')
ax.set_ylabel('Expectation Value')
ax.set_title(f'Expectation Value vs Depth (N={N}, rank={rank})')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogy(ds, diffs, 'o-', color='red')
ax.set_xlabel('HEA Depth')
ax.set_ylabel('|EC - RSS|')
ax.set_title('Absolute Difference')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Expectation value timing vs batch size

In [ ]:
N = 8
rank = 8
depth = 5
shots_list = [1_000, 10_000, 100_000]
batch_sizes = [1, 4, 16, 64]

circuit = build_hea_circuit(N, rank, depth, device=DEVICE)
hamiltonian = build_tfim_hamiltonian(N)

timing_results = []

for B in batch_sizes:
    param_batch = torch.randn(B, circuit.params_size, device=DEVICE)

    # Efficient contraction (shots ignored)
    _, t_ec = time_expectation(
        expectation_value_batch_efficient_contraction,
        param_batch, circuit, hamiltonian, 0
    )

    for shots in shots_list:
        _, t_rss = time_expectation(
            expectation_value_batch_right_suffix,
            param_batch, circuit, hamiltonian, shots
        )
        timing_results.append({'B': B, 'shots': shots, 't_ec': t_ec, 't_rss': t_rss})
        print(f"B={B:3d}  shots={shots:>7d}  EC={t_ec:8.1f}ms  RSS={t_rss:8.1f}ms  speedup={t_rss/t_ec:.1f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for shots in shots_list:
    subset = [r for r in timing_results if r['shots'] == shots]
    bs = [r['B'] for r in subset]
    ts = [r['t_rss'] for r in subset]
    ax.plot(bs, ts, 's--', label=f'RSS {shots//1000}k shots')

# EC is the same regardless of shots
ec_times = {r['B']: r['t_ec'] for r in timing_results}
ax.plot(list(ec_times.keys()), list(ec_times.values()), 'o-', color='black', linewidth=2, label='Efficient Contraction')

ax.set_xlabel('Batch Size (B)')
ax.set_ylabel('Time (ms)')
ax.set_title(f'Expectation Value Time (N={N}, rank={rank}, depth={depth})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 3. Full gradient: accuracy comparison

Compare the full parameter-shift gradient vector from both methods.

In [ ]:
N = 6
rank = 4
depth = 5
shift = torch.pi / 2

circuit = build_hea_circuit(N, rank, depth, device=DEVICE)
hamiltonian = build_tfim_hamiltonian(N)
theta = torch.randn(circuit.params_size, device=DEVICE)
P = theta.numel()

print(f'N={N}, rank={rank}, depth={depth}, params={P}')

# Efficient contraction gradient (exact)
grad_ec = BatchParameterShiftGradient(
    shift=shift, batch_size=P, shots=0,
    measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=depth
)
g_ec = grad_ec.run(theta, circuit, hamiltonian)

# RSS gradient
for shots in [10_000, 100_000]:
    grad_rss = BatchParameterShiftGradient(
        shift=shift, batch_size=P, shots=shots,
        measure_method=MeasureMethod.RIGHT_SUFFIX_SAMPLING, depth=depth
    )
    g_rss = grad_rss.run(theta, circuit, hamiltonian)

    cos_sim = torch.nn.functional.cosine_similarity(g_ec.unsqueeze(0), g_rss.unsqueeze(0)).item()
    mse = ((g_ec - g_rss) ** 2).mean().item()
    max_err = (g_ec - g_rss).abs().max().item()
    print(f'  shots={shots:>7d}  cosine_sim={cos_sim:.6f}  MSE={mse:.2e}  max|err|={max_err:.4f}')

In [ ]:
# Scatter plot: EC gradient vs RSS gradient
grad_rss_100k = BatchParameterShiftGradient(
    shift=shift, batch_size=P, shots=100_000,
    measure_method=MeasureMethod.RIGHT_SUFFIX_SAMPLING, depth=depth
)
g_rss = grad_rss_100k.run(theta, circuit, hamiltonian)

fig, ax = plt.subplots(figsize=(6, 6))
g1 = g_ec.cpu().numpy()
g2 = g_rss.cpu().numpy()
ax.scatter(g1, g2, alpha=0.5, s=10)
lim = max(abs(g1).max(), abs(g2).max()) * 1.1
ax.plot([-lim, lim], [-lim, lim], 'r--', alpha=0.5)
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel('EC gradient (exact)')
ax.set_ylabel('RSS gradient (100k shots)')
ax.set_title(f'Gradient Correlation (N={N}, rank={rank}, depth={depth})')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Full gradient: timing comparison

In [ ]:
configs = [
    {'N': 6, 'rank': 4, 'depth': 5},
    {'N': 8, 'rank': 4, 'depth': 5},
    {'N': 8, 'rank': 8, 'depth': 5},
    {'N': 8, 'rank': 4, 'depth': 10},
    {'N': 10, 'rank': 4, 'depth': 5},
]

grad_timing = []

for cfg in configs:
    N, rank, depth = cfg['N'], cfg['rank'], cfg['depth']
    circuit = build_hea_circuit(N, rank, depth, device=DEVICE)
    hamiltonian = build_tfim_hamiltonian(N)
    theta = torch.randn(circuit.params_size, device=DEVICE)
    P = theta.numel()

    label = f'N={N} r={rank} d={depth} P={P}'

    grad_ec = BatchParameterShiftGradient(
        shift=torch.pi/2, batch_size=None, shots=0,
        measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=depth
    )
    _, t_ec = time_gradient(grad_ec, theta, circuit, hamiltonian, warmup=1, repeats=3)

    for shots in [10_000, 100_000]:
        grad_rss = BatchParameterShiftGradient(
            shift=torch.pi/2, batch_size=None, shots=shots,
            measure_method=MeasureMethod.RIGHT_SUFFIX_SAMPLING, depth=depth
        )
        _, t_rss = time_gradient(grad_rss, theta, circuit, hamiltonian, warmup=1, repeats=3)

        grad_timing.append({'label': label, 'shots': shots, 't_ec': t_ec, 't_rss': t_rss})
        print(f'{label:30s}  shots={shots:>7d}  EC={t_ec:8.0f}ms  RSS={t_rss:8.0f}ms')

In [ ]:
# Bar chart comparison
labels_100k = [r for r in grad_timing if r['shots'] == 100_000]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(labels_100k))
width = 0.35

ax.bar(x - width/2, [r['t_ec']/1000 for r in labels_100k], width, label='Efficient Contraction')
ax.bar(x + width/2, [r['t_rss']/1000 for r in labels_100k], width, label='RSS (100k shots)')

ax.set_ylabel('Gradient Time (s)')
ax.set_title('Full Gradient Computation Time')
ax.set_xticks(x)
ax.set_xticklabels([r['label'] for r in labels_100k], rotation=30, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. RSS accuracy vs shots (convergence)

In [ ]:
N = 8
rank = 4
depth = 5

circuit = build_hea_circuit(N, rank, depth, device=DEVICE)
hamiltonian = build_tfim_hamiltonian(N)
theta = torch.randn(circuit.params_size, device=DEVICE)
P = theta.numel()

# Reference: EC gradient
grad_ec = BatchParameterShiftGradient(
    shift=torch.pi/2, batch_size=P, shots=0,
    measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=depth
)
g_ref = grad_ec.run(theta, circuit, hamiltonian)

shot_list = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]
convergence = []

for shots in shot_list:
    mses = []
    cosines = []
    for trial in range(3):
        grad_rss = BatchParameterShiftGradient(
            shift=torch.pi/2, batch_size=P, shots=shots,
            measure_method=MeasureMethod.RIGHT_SUFFIX_SAMPLING, depth=depth
        )
        g_rss = grad_rss.run(theta, circuit, hamiltonian)
        mses.append(((g_ref - g_rss) ** 2).mean().item())
        cosines.append(torch.nn.functional.cosine_similarity(g_ref.unsqueeze(0), g_rss.unsqueeze(0)).item())

    convergence.append({
        'shots': shots,
        'mse': np.mean(mses),
        'mse_std': np.std(mses),
        'cosine': np.mean(cosines),
    })
    print(f"shots={shots:>7d}  MSE={np.mean(mses):.2e}  cosine={np.mean(cosines):.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

shots_arr = [c['shots'] for c in convergence]

ax = axes[0]
ax.loglog(shots_arr, [c['mse'] for c in convergence], 'o-')
# Reference: 1/shots line
s0, m0 = convergence[0]['shots'], convergence[0]['mse']
ax.loglog(shots_arr, [m0 * s0 / s for s in shots_arr], 'r--', alpha=0.5, label='1/shots')
ax.set_xlabel('Shots')
ax.set_ylabel('MSE vs exact')
ax.set_title('RSS Gradient MSE Convergence')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogx(shots_arr, [c['cosine'] for c in convergence], 'o-')
ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.5)
ax.set_xlabel('Shots')
ax.set_ylabel('Cosine Similarity vs exact')
ax.set_title('RSS Gradient Direction Convergence')
ax.set_ylim(None, 1.02)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()